In [0]:
%sql
WITH diabetes_patients AS (
  SELECT 
    d.claim_id, 
    d.patient_id, 
    AVG(p.line_charge) AS avg_line_charge, 
    AVG(p.line_allowed) AS avg_line_allowed
  FROM 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`diagnosis` d
  JOIN 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`procedure` p
    ON d.claim_id = p.claim_id AND d.patient_id = p.patient_id
  WHERE 
    (d.diagnosis_code LIKE 'E10%' OR d.diagnosis_code LIKE 'E11%')
    AND p.line_charge IS NOT NULL
    AND p.line_allowed IS NOT NULL
  GROUP BY 
    d.claim_id, d.patient_id
),
claim_payer AS (
  SELECT DISTINCT
    claim_id, 
    patient_id, 
    pay_type
  FROM 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`medical_claim`
  WHERE 
    pay_type IS NOT NULL
),
patient_demographics AS (
  SELECT DISTINCT
    patient_id, 
    patient_year_of_birth, 
    pay_type
  FROM 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`enrollment`
  WHERE 
    patient_year_of_birth IS NOT NULL
    AND pay_type IS NOT NULL
)
SELECT 
  COALESCE(cp.pay_type, pd.pay_type) AS payer_type,
  CASE 
    WHEN (2024 - pd.patient_year_of_birth) < 18 THEN '0-17'
    WHEN (2024 - pd.patient_year_of_birth) BETWEEN 18 AND 34 THEN '18-34'
    WHEN (2024 - pd.patient_year_of_birth) BETWEEN 35 AND 49 THEN '35-49'
    WHEN (2024 - pd.patient_year_of_birth) BETWEEN 50 AND 64 THEN '50-64'
    WHEN (2024 - pd.patient_year_of_birth) >= 65 THEN '65+'
    ELSE 'Unknown'
  END AS age_group,
  COUNT(DISTINCT dp.claim_id) AS claim_count,
  COUNT(DISTINCT dp.patient_id) AS patient_count,
  AVG(dp.avg_line_charge) AS avg_claim_cost_charge,
  AVG(dp.avg_line_allowed) AS avg_claim_cost_allowed,
  SUM(dp.avg_line_charge) AS total_cost_charge,
  SUM(dp.avg_line_allowed) AS total_cost_allowed
FROM 
  diabetes_patients dp
LEFT JOIN 
  claim_payer cp
  ON dp.claim_id = cp.claim_id AND dp.patient_id = cp.patient_id
LEFT JOIN 
  patient_demographics pd
  ON dp.patient_id = pd.patient_id
WHERE 
  (cp.pay_type IS NOT NULL OR pd.pay_type IS NOT NULL)
  AND pd.patient_year_of_birth IS NOT NULL
GROUP BY 
  payer_type, age_group
ORDER BY 
  payer_type, age_group;

In [0]:
%pip install databricks-sql-connector

In [0]:
%restart_python

In [0]:
%python
from databricks import sql
import pandas as pd

# Get workspace URL from spark context
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

# Get the SQL query from cell 1
sql_query = """
WITH diabetes_patients AS (
  SELECT 
    d.claim_id, 
    d.patient_id, 
    AVG(p.line_charge) AS avg_line_charge, 
    AVG(p.line_allowed) AS avg_line_allowed
  FROM 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`diagnosis` d
  JOIN 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`procedure` p
    ON d.claim_id = p.claim_id AND d.patient_id = p.patient_id
  WHERE 
    (d.diagnosis_code LIKE 'E10%' OR d.diagnosis_code LIKE 'E11%')
    AND p.line_charge IS NOT NULL
    AND p.line_allowed IS NOT NULL
  GROUP BY 
    d.claim_id, d.patient_id
),
claim_payer AS (
  SELECT DISTINCT
    claim_id, 
    patient_id, 
    pay_type
  FROM 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`medical_claim`
  WHERE 
    pay_type IS NOT NULL
),
patient_demographics AS (
  SELECT DISTINCT
    patient_id, 
    patient_year_of_birth, 
    pay_type
  FROM 
    `healthverity_claims_sample_patient_dataset`.`hv_claims_sample`.`enrollment`
  WHERE 
    patient_year_of_birth IS NOT NULL
    AND pay_type IS NOT NULL
)
SELECT 
  COALESCE(cp.pay_type, pd.pay_type) AS payer_type,
  CASE 
    WHEN (2024 - pd.patient_year_of_birth) < 18 THEN '0-17'
    WHEN (2024 - pd.patient_year_of_birth) BETWEEN 18 AND 34 THEN '18-34'
    WHEN (2024 - pd.patient_year_of_birth) BETWEEN 35 AND 49 THEN '35-49'
    WHEN (2024 - pd.patient_year_of_birth) BETWEEN 50 AND 64 THEN '50-64'
    WHEN (2024 - pd.patient_year_of_birth) >= 65 THEN '65+'
    ELSE 'Unknown'
  END AS age_group,
  COUNT(DISTINCT dp.claim_id) AS claim_count,
  COUNT(DISTINCT dp.patient_id) AS patient_count,
  AVG(dp.avg_line_charge) AS avg_claim_cost_charge,
  AVG(dp.avg_line_allowed) AS avg_claim_cost_allowed,
  SUM(dp.avg_line_charge) AS total_cost_charge,
  SUM(dp.avg_line_allowed) AS total_cost_allowed
FROM 
  diabetes_patients dp
LEFT JOIN 
  claim_payer cp
  ON dp.claim_id = cp.claim_id AND dp.patient_id = cp.patient_id
LEFT JOIN 
  patient_demographics pd
  ON dp.patient_id = pd.patient_id
WHERE 
  (cp.pay_type IS NOT NULL OR pd.pay_type IS NOT NULL)
  AND pd.patient_year_of_birth IS NOT NULL
GROUP BY 
  payer_type, age_group
ORDER BY 
  payer_type, age_group;
"""

In [0]:
# Get access token using dbutils
access_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Connect to the SQL warehouse
print(f"Connecting to warehouse at {workspace_url}...")
connection = sql.connect(
    server_hostname=workspace_url,
    http_path="/sql/1.0/warehouses/148ccb90800933a1",
    access_token=access_token
)

cursor = connection.cursor()

# Execute the query
print("Executing query...")
cursor.execute(sql_query)

# Fetch all results
results = cursor.fetchall()

# Get column names
columns = [desc[0] for desc in cursor.description]

# Close cursor and connection
cursor.close()
connection.close()

# Convert to pandas DataFrame for better display
df = pd.DataFrame(results, columns=columns)

print(f"\nQuery executed successfully! Retrieved {len(df)} rows.\n")
display(df)